# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [1]:
!ls "/content/drive/MyDrive/Colab Notebooks/DL/dataset"

ls: cannot access '/content/drive/MyDrive/Colab Notebooks/DL/dataset': No such file or directory


In [2]:
from google.colab import drive
drive.mount('/content/drive')
# Get kaggle data stored in google drive, not able to implement Kaggle API key in colab
!cp -r "/content/drive/MyDrive/Colab Notebooks/DL/dataset" "/content/"
# !cp -r "/content/drive/MyDrive/Colab Notebooks/DL/dataset" "/content/"

Mounted at /content/drive


In [3]:
# Uncomment in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml cairosvg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 156.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.

In [4]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0+cu128
CUDA available: True


In [5]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
CONFIG = {
    # "model_name": "unsloth/Qwen3.5-2B-Instruct-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    "model_name": "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    # https://huggingface.co/unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit
    "max_seq_length": 5000, # 2048
    "lora_r": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 10,
    "gradient_accumulation_steps": 8,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 12000,
    "eval_size": 0.02,
    # TODO: MODIFY DIR, CHANGE SAVE OUTPUT LATTER
    # "output_dir": "/kaggle/working/qwen2b_svg_lora",
    "output_dir": "/content/qwen2b_svg_lora",
}

CONFIG

{'model_name': 'unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit',
 'max_seq_length': 5000,
 'lora_r': 16,
 'lora_alpha': 16,
 'learning_rate': 0.0002,
 'num_train_epochs': 2,
 'per_device_train_batch_size': 10,
 'gradient_accumulation_steps': 8,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 12000,
 'eval_size': 0.02,
 'output_dir': '/content/qwen2b_svg_lora'}

In [6]:
!mkdir -p /content/qwen2b_svg_lora

In [7]:
# Data catalog using the resources listed in contest_docs/03_Data_Design.md.
DATASET_CATALOG = {
    "OmniSVG/MMSVG-Icon": {
        "split": "train",
        "prompt_fields": ["description", "keywords", "detail", "prompt", "text"],
        "svg_fields": ["svg", "picosvg", "completion", "target"],
    },
    "xingxm/SVGX-Core-250k": {
        "split": "train",
        "prompt_fields": ["qwen_caption", "blip_caption", "name", "img_analysis", "prompt"],
        "svg_fields": ["svg", "completion", "target"],
    },
    "xingxm/SVGX-SFT-1M": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input", "query"],
        "svg_fields": ["completion", "output", "svg", "response"],
    },
    "thesantatitan/deepseek-svg-dataset": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input"],
        "svg_fields": ["completion", "output", "svg"],
    },
}

# For a first run, keep to 1-2 sources.
ACTIVE_SOURCES = [
    # "xingxm/SVGX-SFT-1M",
    # "OmniSVG/MMSVG-Icon",
]

In [8]:
def _pick_first_non_empty(example, keys):
    for key in keys:
        if key in example and example[key] is not None:
            val = str(example[key]).strip()
            if val:
                return val
    return ""


def to_prompt_svg(example, prompt_fields, svg_fields):
    prompt = _pick_first_non_empty(example, prompt_fields)
    svg = _pick_first_non_empty(example, svg_fields)
    if not svg.lower().startswith("<svg"):
        return {"prompt": "", "svg": ""}
    return {"prompt": prompt, "svg": svg}


def load_source_dataset(dataset_id, cfg, max_samples):
    print(f"Loading {dataset_id} ...")
    ds = load_dataset(dataset_id, split=cfg["split"])
    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))
    ds = ds.map(
        lambda ex: to_prompt_svg(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
        desc=f"normalizing {dataset_id}",
    )
    ds = ds.filter(lambda x: bool(x["prompt"]) and bool(x["svg"]))
    print(f"{dataset_id}: {len(ds)} usable rows")
    return ds

In [9]:
from google.colab import userdata
from huggingface_hub import login

# Get the token from Colab secrets
HF_TOKEN = userdata.get('HF_TOKEN')

# Log in to Hugging Face
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("Token is not set. Please save the token first.")


Successfully logged in to Hugging Face!


In [10]:
# MY DATA LOADING

# def format_svg_sample(prompt: str, svg_code: str) -> str:
#     """Format a prompt-SVG pair into the chat template expected by the model."""
#     messages = [
#         {"role": "system", "content": "You are an expert SVG code generator. Generate clean, valid SVG code based on the user's description."},
#         {"role": "user", "content": prompt},
#         {"role": "assistant", "content": svg_code},
#     ]
#     return tokenizer.apply_chat_template(messages, tokenize=False)

# new_prompt = []
# for r in tqdm(range(train_df.shape[0])):
#     new_prompt.append(format_svg_sample(train_df.loc[r, 'prompt'], train_df.loc[r, 'svg']))
# train_df['text'] = new_prompt

# import pandas as pd
# from tqdm import tqdm
from pathlib import Path

# DATA = Path('./dataset')
DATA = Path('/content/dataset')
# train_df = pd.read_csv(DATA / 'train.csv')
train_df = pd.read_csv(DATA / 'final_df.csv')
# NOTE: DATA LIMIT SUBSET
# train_df = train_df.iloc[0:1000]

from datasets import Dataset
from datasets import load_dataset

# train_dataset = load_dataset("csv", data_files=str(DATA / "train.csv"))
train_dataset = Dataset.from_pandas(train_df)
train_dataset

Dataset({
    features: ['Unnamed: 0', 'id', 'prompt', 'svg'],
    num_rows: 95251
})

In [11]:
# datasets_ok = []
# for source in ACTIVE_SOURCES:
#     try:
#         ds = load_source_dataset(
#             source,
#             DATASET_CATALOG[source],
#             CONFIG["max_train_samples_per_source"],
#         )
#         datasets_ok.append(ds)
#     except Exception as e:
#         print(f"Skipping {source}: {type(e).__name__}: {e}")

# if not datasets_ok:
#     raise RuntimeError("No dataset loaded. Check dataset IDs, internet access, and schema fields.")
# MY DATA

datasets_ok = [train_dataset]

train_raw = datasets_ok[0] if len(datasets_ok) == 1 else concatenate_datasets(datasets_ok)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

Train rows: 93345
Eval rows: 1906


{'Unnamed: 0': 31780,
 'id': 'b760c35be177500fa2193b033833009c',
 'prompt': 'A green square with a circular pattern inside it, surrounded by three other squares.',
 'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#96E8BA" fill-opacity="1.0"  filling="0" d="M160.31199645996094 184.375 L105.31199645996094 184.375 C88.43800354003906 184.375 75.0 170.625 75.0 154.06199645996094 L75.0 145.0 C75.0 128.125 88.75 114.68800354003906 105.31199645996094 114.68800354003906 L160.31199645996094 114.68800354003906 C177.18800354003906 114.68800354003906 190.625 128.43800354003906 190.625 145.0 L190.625 154.06199645996094 C190.625 170.625 176.875 184.375 160.31199645996094 184.375 Z"></path>\n<path fill="#103E26" fill-opacity="1.0"  filling="0" d="M59.375 90.93800354003906 C39.6879997253418 90.93800354003906 23.75 75.0 23.75 55.3129997253418 C23.75 35.625 39.6879997253418 20.0 59.375 20.0 C79.06199645996094 20.0 95.0 35.93800354

In [12]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text.shape)
print(train_text[0].keys())
print(train_text[0]["text"])

Map:   0%|          | 0/93345 [00:00<?, ? examples/s]

Map:   0%|          | 0/1906 [00:00<?, ? examples/s]

(93345, 1)
dict_keys(['text'])
<|im_start|>system
You generate compact, valid SVG markup from user requests. Return only SVG code with a single root <svg> element.<|im_end|>
<|im_start|>user
A green square with a circular pattern inside it, surrounded by three other squares.<|im_end|>
<|im_start|>assistant
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#96E8BA" fill-opacity="1.0"  filling="0" d="M160.31199645996094 184.375 L105.31199645996094 184.375 C88.43800354003906 184.375 75.0 170.625 75.0 154.06199645996094 L75.0 145.0 C75.0 128.125 88.75 114.68800354003906 105.31199645996094 114.68800354003906 L160.31199645996094 114.68800354003906 C177.18800354003906 114.68800354003906 190.625 128.43800354003906 190.625 145.0 L190.625 154.06199645996094 C190.625 170.625 176.875 184.375 160.31199645996094 184.375 Z"></path>
<path fill="#103E26" fill-opacity="1.0"  filling="0" d="M59.375 90.93800354003906 C39.6879997253418 90.938

In [13]:
# # Load Saved Model on Drive
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="/content/drive/MyDrive/Colab_Notebooks/DL/qwen2b_svg_lora",
#     max_seq_length=CONFIG["max_seq_length"],
#     dtype=None,
#     load_in_4bit=True,
# )


In [14]:
# Or Load New Model Instance
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Qwen3_Vl patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.41G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    # num_train_epochs=3,
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    # evaluation_strategy="steps",
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],

    # Early Stopping Implementation
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
    # Early Stopping Implementation
    callbacks=[EarlyStoppingCallback(early_stopping_patience=6)],
)

train_result = trainer.train()
train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/93345 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1906 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 93,345 | Num Epochs = 2 | Total steps = 2,334
O^O/ \_/ \    Batch size per device = 10 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (10 x 8 x 1) = 80
 "-____-"     Trainable parameters = 17,432,576 of 2,144,964,608 (0.81% trained)


Step,Training Loss,Validation Loss
100,0.494580,0.475214
200,0.410175,0.412052
300,0.397351,0.387478
400,0.381520,0.371672
500,0.375771,0.360863
600,0.352411,0.352123
700,0.350086,0.345241
800,0.335672,0.339393
900,0.334774,0.333970
1000,0.334709,0.327927


In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!cp -r "/content/qwen2b_svg_lora" "/content/drive/MyDrive/Colab Notebooks/DL/"


## KAGGLE SUBMISSION

In [ ]:
from unsloth import FastLanguageModel

# LOading Model
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name='/content/drive/MyDrive/Colab_Notebooks/DL/content/qwen2b_svg_lora',
    # model_name=CONFIG['output_dir'],
    model_name="/content/drive/MyDrive/Colab Notebooks/DL/qwen2b_svg_lora",
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)


In [ ]:
FastLanguageModel.for_inference(model)

# ORIGINAL REGEX WRONG!!!
# SVG_REGEX = re.compile(r"<svg[\\s\\S]*?</svg>", flags=re.IGNORECASE)
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)
# MUST NOT USE \\ in re, likely original typo


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def generate_svg(prompt, max_new_tokens=CONFIG['max_seq_length']):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    # print('TEST 3')
    # inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    inputs = tokenizer(text=chat_text, images=None, return_tensors="pt").to(model.device)

    # print('TEST 4')
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
        )
        print(output_ids)
    # print('TEST 5')
    output_ids = output_ids.cpu()
    # print("output_ids")
    # print(output_ids)
    # print(output_ids)
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print("DECODED")
    print(decoded)
    decoded = decoded.replace('<svg>', '(omit-svg)', 1)
    print('TEST 6')
    svg = extract_svg(decoded)
    print('EXTRACTED SVG')
    print(svg)
    print('TEST 7')
    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)
    return svg

In [ ]:
# test_prompt = "a simple blue bird icon"
test_prompt = "Create a blue bird"
pred_svg = generate_svg(test_prompt, max_new_tokens=1000)
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))

In [ ]:
# NOT WHAT WE WANT
from IPython.display import SVG, display, HTML
import cairosvg
from IPython.display import Image
import io

# print(sub_df.loc[0, 'svg'])
display(SVG(pred_svg))

### Load Saved Model in Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from unsloth import FastLanguageModel

# LOading Model
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name='/content/drive/MyDrive/Colab_Notebooks/DL/content/qwen2b_svg_lora',
    # model_name=CONFIG['output_dir'],
    model_name="/content/drive/MyDrive/Colab Notebooks/DL/qwen2b_svg_lora",
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)


FastLanguageModel.for_inference(model)

In [ ]:
import gc
gc.collect()
# Clear PyTorch's GPU memory cache
torch.cuda.empty_cache()

import tensorflow as tf
tf.keras.backend.clear_session()

# del model


In [ ]:
from tqdm import tqdm

def generate_svg_batch(all_prompts:list[str], max_new_tokens=CONFIG['max_seq_length'], batchsize=8):
    all_svgs = []

    for p_i in tqdm(range(0, len(all_prompts), batchsize), desc='Svg To Prompts'):
        print('Converting Prompt')
        chat_texts = [
            "<|im_start|>system\n"
            f"{SYSTEM_PROMPT}<|im_end|>\n"
            "<|im_start|>user\n"
            f"{prompt}<|im_end|>\n"
            "<|im_start|>assistant\n" for prompt in all_prompts[p_i : p_i + batchsize]
        ]

        print('Running Tokenizer')
        inputs = tokenizer(
            text=chat_texts,
            padding=True,
            return_tensors="pt",
            padding_side='left', # TODO: UNDERSTAND PADDING EFFECT
        ).to(model.device)

        print('Running Model')
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.05,
            )
            # print(output_ids.shape)
        output_ids = output_ids.cpu() # Move output to cpu for tokenizer

        print('Tokenizer Decode')
        decoded_prompts = tokenizer.decode(output_ids, skip_special_tokens=True)

        decoded_prompts = [decoded.replace('<svg>', '(omit-svg)', 1) for decoded in decoded_prompts]
        # Done due to <svg> inclusion in the prompt, which causes regex parsing errors, and then defaults to default svg output
        # May be ommited, if there is a different model used that purely returns svg code

        for i in range(len(decoded_prompts)):
            svg = extract_svg(decoded_prompts[i])
            # print(f"extracted svg len {len(svg)}")
            if not is_valid_svg(svg):
                svg = fallback_svg( 'NULL' )
                # print(f"fallback svg len {len(svg)}")
            all_svgs.append(svg)

    return all_svgs

In [ ]:
TEST_PROMPTS_PATH = "./dataset/test.csv"
SUBMISSION_PATH = "./dataset/sample_submission.csv"

test_df = pd.read_csv(TEST_PROMPTS_PATH)
# test_df = test_df.iloc[0:10]

rows = []
invalid_count = 0
t0 = time.time()

all_svgs = generate_svg_batch(
    test_df['prompt'].tolist(),
    batchsize=60
)
print(len(all_svgs))
print(all_svgs)
print(test_df.shape)
for i in range(len(all_svgs)):
    rows.append({"id": test_df.loc[i, "id"], "svg": all_svgs[i]})

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()

In [ ]:
sub_df.head(10)

In [ ]:
test_df['prompt'].head(10), test_df['prompt'].iloc[6]

## Save Results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# !cp -r "/content/qwen2b_svg_lora" "/content/drive/MyDrive/Colab_Notebooks/DL"


In [ ]:
!cp -r "./dataset/sample_submission.csv" "/content/drive/MyDrive/Colab Notebooks/DL"

## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.

In [ ]:
from google.colab import runtime
runtime.unassign()
